# Reproject All Frames to First Frame

This notebook investigates reprojecting all frames' point clouds into the first frame instead of the previous frame.
It visualizes:
- Reprojected points on RGB and depth images of the first frame
- Error statistics (depth errors, validity)
- Comparison across multiple frames

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
from typing import Optional, Tuple, Dict, List

# Add project root to path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

# Import project modules
from point2pose.io.sources.dataset.datareader import Ho3dReader, YcbineoatReader
from point2pose.data_types.frame import Frame
from point2pose.utils.camera import (
    extract_cropped_point_cloud,
    project_points_to_image,
    compute_projection_consistency,
)
from point2pose.utils.transform import inverse_SE3

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Configuration
ho3d_root = "/home/justin/data/HO3D_V3/"  # Root directory of HO3D dataset
video_dir = "/home/justin/data/HO3D_V3/evaluation/MPM10"  # Path to specific video sequence

# Optional: Path to meta_data.npz for estimated poses
meta_data_path = None  # Set to path like "/path/to/meta_data/meta_data.npz" or None to auto-detect
results_dir = "/home/justin/code/point-to-pose/results/ho3d_single"  # For auto-detection
video_name = "MPM10"  # For auto-detection

# Frame indices to analyze (0 is the first frame, target for reprojection)
first_frame_idx = 0
frame_indices_to_analyze = [272,300,400,500,600,700]  # Frames to reproject into first frame

obj_id = 0  # Object ID
min_depth = 0.05
max_depth = 0.8

In [ ]:
# Initialize reader
reader = Ho3dReader(video_dir, ho3d_root)
print(f"Loaded {len(reader)} frames from {reader.get_video_name()}")
print(f"Camera intrinsics:\n{reader.K}")

In [ ]:
def load_frame_from_reader(reader, frame_idx: int) -> Optional[Frame]:
    """Load a Frame object from Ho3dReader."""
    try:
        rgb = cv2.imread(reader.color_files[frame_idx])
        if rgb is None:
            return None
        H, W = rgb.shape[:2]
        rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)
        depth = reader.get_depth(frame_idx)
        mask = reader.get_mask(frame_idx)
        mask = cv2.resize(mask, (W, H), interpolation=cv2.INTER_NEAREST)
        mask_tensor = torch.from_numpy(mask).float().unsqueeze(0).unsqueeze(0).to(device)
        
        return Frame(
            id=frame_idx,
            rgb=rgb,
            depth=depth,
            mask=mask_tensor,
            intrinsics=reader.K,
            depth_factor=1.0,  # HO3D depth is already in meters
        )
    except Exception as e:
        print(f"Error loading frame {frame_idx}: {e}")
        return None

# Load first frame (target frame for reprojection)
first_frame = load_frame_from_reader(reader, first_frame_idx)
if first_frame is None:
    raise ValueError(f"Failed to load first frame (index {first_frame_idx})")

print(f"Loaded first frame (index {first_frame_idx})")
print(f"  RGB shape: {first_frame.rgb.shape}")
print(f"  Depth shape: {first_frame.depth.shape}")
print(f"  Mask shape: {first_frame.mask.shape if first_frame.mask is not None else None}")

In [ ]:
def compute_per_point_reprojection_errors(
    src_pcd: np.ndarray,
    T_src2dst: np.ndarray,
    frame_dst: Frame,
    obj_id: int = 0,
    min_depth: float = 0.01,
    max_depth: float = 2.0,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Compute per-point reprojection errors.
    
    Returns:
        pts_2d: (N, 2) projected 2D coordinates
        depth_errors: (N,) per-point depth errors (NaN for invalid points)
        valid_mask: (N,) boolean mask indicating valid points
        pts_3d_dst: (N, 3) 3D points in destination frame
        invalid_reasons: (N,) integer array: 0=valid, 1=out_of_bounds, 2=not_in_mask, 3=no_depth, 4=depth_out_of_range
    """
    # Project points
    pts_dst_2d, pts_dst_3d = project_points_to_image(
        src_pcd, frame_dst.intrinsics, T_src2dst
    )

    # Get frame properties
    H, W = frame_dst.depth.shape
    if frame_dst.mask is not None:
        if isinstance(frame_dst.mask, torch.Tensor):
            dst_mask = frame_dst.mask[obj_id, 0].cpu().numpy()
        else:
            dst_mask = np.asarray(frame_dst.mask[obj_id, 0])
    else:
        dst_mask = np.ones((H, W), dtype=bool)
    depth_image = frame_dst.depth
    depth_factor = frame_dst.depth_factor if frame_dst.depth_factor is not None else 1.0

    N = len(pts_dst_2d)

    # Convert 2D coordinates to integer pixel indices
    u_coords = np.round(pts_dst_2d[:, 0]).astype(int)
    v_coords = np.round(pts_dst_2d[:, 1]).astype(int)

    # Check bounds
    in_bounds = (u_coords >= 0) & (u_coords < W) & (v_coords >= 0) & (v_coords < H)

    # Get mask values for valid points
    mask_values = np.zeros(N, dtype=bool)
    valid_indices = np.where(in_bounds)[0]
    if len(valid_indices) > 0:
        mask_values[valid_indices] = (
            dst_mask[v_coords[valid_indices], u_coords[valid_indices]] > 0
        )

    # Get measured depths
    z_measured = np.full(N, np.nan, dtype=float)
    if len(valid_indices) > 0:
        z_measured[valid_indices] = (
            depth_image[v_coords[valid_indices], u_coords[valid_indices]] / depth_factor
        )

    # Get projected depths
    z_projected = pts_dst_3d[:, 2]

    # Create validity mask: inside mask, valid depth, in range
    valid_depth = (z_measured > 0) & np.isfinite(z_measured)
    in_depth_range = (
        (min_depth <= z_measured)
        & (z_measured <= max_depth)
        & (min_depth <= z_projected)
        & (z_projected <= max_depth)
    )

    valid = mask_values & valid_depth & in_depth_range

    # Compute depth errors (NaN for invalid points)
    depth_errors = np.full(N, np.nan, dtype=float)
    depth_errors[valid] = np.abs(z_projected[valid] - z_measured[valid])

    # Classify invalid reasons
    invalid_reasons = np.zeros(N, dtype=int)
    invalid_reasons[~in_bounds] = 1  # out_of_bounds
    invalid_reasons[in_bounds & ~mask_values] = 2  # not_in_mask
    invalid_reasons[in_bounds & mask_values & ~valid_depth] = 3  # no_depth
    invalid_reasons[in_bounds & mask_values & valid_depth & ~in_depth_range] = 4  # depth_out_of_range
    invalid_reasons[valid] = 0  # valid points

    return pts_dst_2d, depth_errors, valid, pts_dst_3d, invalid_reasons

In [ ]:
# Load estimated poses from meta_data if available
def find_meta_data_path(meta_data_path_override=None, results_dir=None, video_name=None):
    """Find meta_data.npz file in expected locations."""
    if meta_data_path_override and os.path.exists(meta_data_path_override):
        return meta_data_path_override
    
    project_root = os.path.abspath('..')
    candidates = []
    
    if results_dir and video_name:
        candidates.append(os.path.join(results_dir, video_name, "meta_data", "meta_data.npz"))
        candidates.append(os.path.join(project_root, "results", "ho3d_single", video_name, "meta_data", "meta_data.npz"))
    
    candidates.extend([
        os.path.join(project_root, "meta_data", "meta_data.npz"),
        os.path.join(project_root, "debug", "pipeline", "meta_data", "meta_data.npz"),
        os.path.join(os.getcwd(), "meta_data", "meta_data.npz"),
    ])
    
    for path in candidates:
        if path and os.path.exists(path):
            print(f"Found meta_data.npz at: {path}")
            return path
    
    return None

def load_estimated_poses(meta_data_path):
    """Load estimated poses from meta_data.npz."""
    if meta_data_path is None or not os.path.exists(meta_data_path):
        return None, None
    
    try:
        data = np.load(meta_data_path, allow_pickle=True)
        
        # Get frame IDs
        frame_ids = data.get("frame_id", None)
        if frame_ids is None:
            print("Warning: No frame_id found in meta_data")
            return None, None
        
        # Get estimated poses
        obj_pose = data.get("obj_pose", None)
        if obj_pose is None:
            print("Warning: No obj_pose found in meta_data")
            return None, None
        
        # Handle different storage formats
        poses_dict = {}
        if obj_pose.dtype == object:
            # Object array - convert to dict
            for i, pose in enumerate(obj_pose):
                if pose is not None:
                    poses_dict[i] = np.asarray(pose, dtype=float)
        else:
            # Fixed shape array (N, 4, 4) or (N, 16)
            if obj_pose.ndim == 3 and obj_pose.shape[1:] == (4, 4):
                for i in range(len(obj_pose)):
                    poses_dict[i] = np.asarray(obj_pose[i], dtype=float)
            elif obj_pose.ndim == 2 and obj_pose.shape[1] == 16:
                for i in range(len(obj_pose)):
                    poses_dict[i] = np.asarray(obj_pose[i].reshape(4, 4), dtype=float)
        
        print(f"Loaded {len(poses_dict)} estimated poses from meta_data")
        return poses_dict, frame_ids
    except Exception as e:
        print(f"Error loading estimated poses: {e}")
        import traceback
        traceback.print_exc()
        return None, None

# Try to load estimated poses
meta_data_file = find_meta_data_path(meta_data_path, results_dir, video_name)
estimated_poses, meta_frame_ids = load_estimated_poses(meta_data_file)

# Reproject all specified frames into the first frame
results = {}
results_estimated = {}  # Results using estimated poses

for frame_idx in frame_indices_to_analyze:
    if frame_idx >= len(reader):
        print(f"Skipping frame {frame_idx} (out of range)")
        continue
    
    print(f"\nProcessing frame {frame_idx}...")
    
    # Load current frame
    curr_frame = load_frame_from_reader(reader, frame_idx)
    if curr_frame is None:
        print(f"  Failed to load frame {frame_idx}")
        continue
    
    # Extract full point cloud from current frame
    src_pcd_full = extract_cropped_point_cloud(curr_frame, obj_id)
    if src_pcd_full.size == 0:
        print(f"  No points extracted from frame {frame_idx}")
        continue
    
    print(f"  Extracted {len(src_pcd_full)} points from frame {frame_idx}")
    
    # Get GT poses for both frames
    gt_pose_curr = reader.get_gt_pose(frame_idx)
    gt_pose_first = reader.get_gt_pose(first_frame_idx)
    
    if gt_pose_curr is None or gt_pose_first is None:
        print(f"  Warning: GT poses not available for frame {frame_idx} or first frame")
        # Use identity transform as fallback
        T_curr2first = np.eye(4)
    else:
        # Compute transform from current frame to first frame
        # T_first_curr = T_first_obj @ inv(T_curr_obj)
        T_curr2first = gt_pose_first @ inverse_SE3(gt_pose_curr)
    
    # Compute reprojection errors
    pts_2d, depth_errors, valid_mask, pts_dst_3d, invalid_reasons = compute_per_point_reprojection_errors(
        src_pcd_full,
        T_curr2first,
        first_frame,
        obj_id=obj_id,
        min_depth=min_depth,
        max_depth=max_depth,
    )
    
    # Compute statistics
    valid_errors = depth_errors[valid_mask]
    if np.isfinite(valid_errors).any() and len(valid_errors) > 0:
        mean_error = float(np.nanmean(valid_errors))
        median_error = float(np.nanmedian(valid_errors))
        std_error = float(np.nanstd(valid_errors))
        n_valid = int(np.sum(valid_mask))
    else:
        mean_error = np.nan
        median_error = np.nan
        std_error = np.nan
        n_valid = 0
    
    # Count invalid reasons
    invalid_counts = {
        'out_of_bounds': int(np.sum(invalid_reasons == 1)),
        'not_in_mask': int(np.sum(invalid_reasons == 2)),
        'no_depth': int(np.sum(invalid_reasons == 3)),
        'depth_out_of_range': int(np.sum(invalid_reasons == 4)),
    }
    
    results[frame_idx] = {
        'pts_2d': pts_2d,
        'depth_errors': depth_errors,
        'valid_mask': valid_mask,
        'invalid_reasons': invalid_reasons,
        'src_pcd_full': src_pcd_full,
        'T_curr2first': T_curr2first,
        'mean_error': mean_error,
        'median_error': median_error,
        'std_error': std_error,
        'n_valid': n_valid,
        'n_total': len(src_pcd_full),
        'invalid_counts': invalid_counts,
    }
    
    print(f"  Valid points: {n_valid}/{len(src_pcd_full)} ({100*n_valid/len(src_pcd_full):.1f}%)")
    print(f"  Mean depth error: {mean_error:.4f}m")
    print(f"  Median depth error: {median_error:.4f}m")
    print(f"  Invalid breakdown: {invalid_counts}")
    
    # Also reproject using estimated poses if available
    if estimated_poses is not None:
        est_pose_curr = estimated_poses.get(frame_idx)
        est_pose_first = estimated_poses.get(first_frame_idx)
        
        if est_pose_curr is not None and est_pose_first is not None:
            # Compute transform using estimated poses
            T_curr2first_est = est_pose_first @ inverse_SE3(est_pose_curr)
            
            # Compute reprojection errors with estimated poses
            pts_2d_est, depth_errors_est, valid_mask_est, pts_dst_3d_est, invalid_reasons_est = compute_per_point_reprojection_errors(
                src_pcd_full,
                T_curr2first_est,
                first_frame,
                obj_id=obj_id,
                min_depth=min_depth,
                max_depth=max_depth,
            )
            
            # Compute statistics for estimated
            valid_errors_est = depth_errors_est[valid_mask_est]
            if np.isfinite(valid_errors_est).any() and len(valid_errors_est) > 0:
                mean_error_est = float(np.nanmean(valid_errors_est))
                median_error_est = float(np.nanmedian(valid_errors_est))
                std_error_est = float(np.nanstd(valid_errors_est))
                n_valid_est = int(np.sum(valid_mask_est))
            else:
                mean_error_est = np.nan
                median_error_est = np.nan
                std_error_est = np.nan
                n_valid_est = 0
            
            invalid_counts_est = {
                'out_of_bounds': int(np.sum(invalid_reasons_est == 1)),
                'not_in_mask': int(np.sum(invalid_reasons_est == 2)),
                'no_depth': int(np.sum(invalid_reasons_est == 3)),
                'depth_out_of_range': int(np.sum(invalid_reasons_est == 4)),
            }
            
            results_estimated[frame_idx] = {
                'pts_2d': pts_2d_est,
                'depth_errors': depth_errors_est,
                'valid_mask': valid_mask_est,
                'invalid_reasons': invalid_reasons_est,
                'T_curr2first': T_curr2first_est,
                'mean_error': mean_error_est,
                'median_error': median_error_est,
                'std_error': std_error_est,
                'n_valid': n_valid_est,
                'n_total': len(src_pcd_full),
                'invalid_counts': invalid_counts_est,
            }
            
            print(f"  [ESTIMATED] Valid: {n_valid_est}/{len(src_pcd_full)} ({100*n_valid_est/len(src_pcd_full):.1f}%)")
            print(f"  [ESTIMATED] Mean error: {mean_error_est:.4f}m, Median: {median_error_est:.4f}m")
        else:
            print(f"  Warning: Estimated poses not available for frame {frame_idx} or first frame")

In [ ]:
def plot_reprojection_visualization(
    frame_dst: Frame,
    pts_2d: np.ndarray,
    depth_errors: np.ndarray,
    valid_mask: np.ndarray,
    invalid_reasons: np.ndarray,
    frame_idx: int,
    stats: Dict,
    title_suffix: str = "",
) -> plt.Figure:
    """Visualize reprojected points on RGB and depth images."""
    
    rgb = frame_dst.rgb.copy()
    if rgb.dtype != np.uint8:
        rgb = (rgb * 255).astype(np.uint8) if rgb.max() <= 1.0 else rgb.astype(np.uint8)
    
    # Get depth image for visualization
    depth_image = None
    if frame_dst.depth is not None:
        depth_image = frame_dst.depth.copy()
        depth_factor = frame_dst.depth_factor if frame_dst.depth_factor is not None else 1.0
        # Normalize depth for visualization
        if depth_image.dtype != np.uint8:
            depth_normalized = depth_image / depth_factor
            depth_normalized = np.clip(
                (depth_normalized - depth_normalized.min()) / (depth_normalized.max() - depth_normalized.min() + 1e-8),
                0, 1
            )
            depth_image = (depth_normalized * 255).astype(np.uint8)
    
    # Separate valid and invalid points
    valid_pts = pts_2d[valid_mask]
    valid_errors = depth_errors[valid_mask]
    invalid_pts = pts_2d[~valid_mask]
    invalid_reasons_array = invalid_reasons[~valid_mask]
    
    # Create figure with three subplots
    if depth_image is not None:
        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(24, 8))
    else:
        fig, (ax1, ax3) = plt.subplots(1, 2, figsize=(16, 8))
        ax2 = None
    
    # Plot 1: RGB image with reprojected points
    ax1.imshow(rgb)
    
    # Plot invalid points with different colors based on reason
    if len(invalid_pts) > 0:
        out_of_bounds = invalid_pts[invalid_reasons_array == 1]
        not_in_mask = invalid_pts[invalid_reasons_array == 2]
        no_depth = invalid_pts[invalid_reasons_array == 3]
        depth_out_of_range = invalid_pts[invalid_reasons_array == 4]
        
        if len(out_of_bounds) > 0:
            ax1.scatter(out_of_bounds[:, 0], out_of_bounds[:, 1], c="red", s=4, alpha=0.4, marker="x",
                       label=f"Out of bounds ({len(out_of_bounds)})")
        if len(not_in_mask) > 0:
            ax1.scatter(not_in_mask[:, 0], not_in_mask[:, 1], c="yellow", s=4, alpha=0.4, marker="x",
                       label=f"Not in mask ({len(not_in_mask)})")
        if len(no_depth) > 0:
            ax1.scatter(no_depth[:, 0], no_depth[:, 1], c="magenta", s=4, alpha=0.4, marker="x",
                       label=f"No depth ({len(no_depth)})")
        if len(depth_out_of_range) > 0:
            ax1.scatter(depth_out_of_range[:, 0], depth_out_of_range[:, 1], c="cyan", s=4, alpha=0.4, marker="x",
                       label=f"Depth out of range ({len(depth_out_of_range)})")
    
    # Plot valid points colored by error
    if len(valid_pts) > 0:
        if np.isfinite(valid_errors).any():
            error_min = np.nanmin(valid_errors)
            error_max = np.nanmax(valid_errors)
            if error_max > error_min:
                normalized_errors = (valid_errors - error_min) / (error_max - error_min)
            else:
                normalized_errors = np.zeros_like(valid_errors)
        else:
            normalized_errors = np.zeros_like(valid_errors)
        
        scatter = ax1.scatter(
            valid_pts[:, 0], valid_pts[:, 1],
            c=normalized_errors, cmap="RdYlGn_r",
            s=8, alpha=0.6, vmin=0, vmax=1,
            label=f"Valid ({len(valid_pts)})"
        )
        plt.colorbar(scatter, ax=ax1, label="Normalized depth error")
    
    ax1.set_title(f"RGB: Frame {frame_idx} → First Frame{title_suffix}\n"
                  f"Total: {stats['n_total']}, Valid: {stats['n_valid']} ({100*stats['n_valid']/stats['n_total']:.1f}%)")
    ax1.set_xlabel("u (pixels)")
    ax1.set_ylabel("v (pixels)")
    ax1.legend(loc="best", fontsize=8)
    
    # Plot 2: Depth image with reprojected points
    if ax2 is not None and depth_image is not None:
        ax2.imshow(depth_image, cmap="gray")
        
        # Plot invalid points
        if len(invalid_pts) > 0:
            out_of_bounds = invalid_pts[invalid_reasons_array == 1]
            not_in_mask = invalid_pts[invalid_reasons_array == 2]
            no_depth = invalid_pts[invalid_reasons_array == 3]
            depth_out_of_range = invalid_pts[invalid_reasons_array == 4]
            
            if len(out_of_bounds) > 0:
                ax2.scatter(out_of_bounds[:, 0], out_of_bounds[:, 1], c="red", s=4, alpha=0.4, marker="x")
            if len(not_in_mask) > 0:
                ax2.scatter(not_in_mask[:, 0], not_in_mask[:, 1], c="yellow", s=4, alpha=0.4, marker="x")
            if len(no_depth) > 0:
                ax2.scatter(no_depth[:, 0], no_depth[:, 1], c="magenta", s=4, alpha=0.4, marker="x")
            if len(depth_out_of_range) > 0:
                ax2.scatter(depth_out_of_range[:, 0], depth_out_of_range[:, 1], c="cyan", s=4, alpha=0.4, marker="x")
        
        # Plot valid points
        if len(valid_pts) > 0:
            if np.isfinite(valid_errors).any():
                error_min = np.nanmin(valid_errors)
                error_max = np.nanmax(valid_errors)
                if error_max > error_min:
                    normalized_errors = (valid_errors - error_min) / (error_max - error_min)
                else:
                    normalized_errors = np.zeros_like(valid_errors)
            else:
                normalized_errors = np.zeros_like(valid_errors)
            
            ax2.scatter(
                valid_pts[:, 0], valid_pts[:, 1],
                c=normalized_errors, cmap="RdYlGn_r",
                s=8, alpha=0.6, vmin=0, vmax=1
            )
        
        ax2.set_title("Depth: Reprojected points overlay")
        ax2.set_xlabel("u (pixels)")
        ax2.set_ylabel("v (pixels)")
    
    # Plot 3: Error histogram
    if len(valid_pts) > 0 and np.isfinite(valid_errors).any():
        ax3.hist(valid_errors, bins=50, alpha=0.7, color="tab:blue", edgecolor="black")
        ax3.axvline(stats['mean_error'], color="red", linestyle="--", linewidth=2,
                   label=f"Mean: {stats['mean_error']:.4f}m")
        ax3.axvline(stats['median_error'], color="green", linestyle="--", linewidth=2,
                   label=f"Median: {stats['median_error']:.4f}m")
        ax3.set_xlabel("Depth error (meters)")
        ax3.set_ylabel("Frequency")
        ax3.set_title(f"Depth error distribution (valid points: {stats['n_valid']})")
        ax3.legend()
        ax3.grid(True, alpha=0.3)
    else:
        ax3.text(0.5, 0.5, f"No valid errors\nTotal: {stats['n_total']}\nValid: {stats['n_valid']}\nInvalid: {stats['n_total'] - stats['n_valid']}",
                ha="center", va="center", transform=ax3.transAxes)
        ax3.set_title("Depth error distribution")
    
    fig.tight_layout()
    return fig

In [ ]:
# Visualize reprojection for each frame (GT poses)
print("\n" + "="*80)
print("VISUALIZING REPROJECTION USING GT POSES")
print("="*80)
for frame_idx, result in results.items():
    fig = plot_reprojection_visualization(
        first_frame,
        result['pts_2d'],
        result['depth_errors'],
        result['valid_mask'],
        result['invalid_reasons'],
        frame_idx,
        {
            'mean_error': result['mean_error'],
            'median_error': result['median_error'],
            'n_valid': result['n_valid'],
            'n_total': result['n_total'],
        },
        title_suffix=" (GT poses)",
    )
    plt.show()

# Visualize reprojection for each frame (Estimated poses)
if results_estimated:
    print("\n" + "="*80)
    print("VISUALIZING REPROJECTION USING ESTIMATED POSES")
    print("="*80)
    for frame_idx, result in results_estimated.items():
        fig = plot_reprojection_visualization(
            first_frame,
            result['pts_2d'],
            result['depth_errors'],
            result['valid_mask'],
            result['invalid_reasons'],
            frame_idx,
            {
                'mean_error': result['mean_error'],
                'median_error': result['median_error'],
                'n_valid': result['n_valid'],
                'n_total': result['n_total'],
            },
            title_suffix=" (Estimated poses)",
        )
        plt.show()

In [ ]:
# Summary statistics across all frames
print("\n" + "="*80)
print("SUMMARY STATISTICS: Reprojection to First Frame (GT Poses)")
print("="*80)

frame_indices = sorted(results.keys())
mean_errors = [results[idx]['mean_error'] for idx in frame_indices]
median_errors = [results[idx]['median_error'] for idx in frame_indices]
valid_ratios = [results[idx]['n_valid'] / results[idx]['n_total'] for idx in frame_indices]
n_valid_list = [results[idx]['n_valid'] for idx in frame_indices]
n_total_list = [results[idx]['n_total'] for idx in frame_indices]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Mean error over time
axes[0, 0].plot(frame_indices, mean_errors, 'o-', color='tab:blue', linewidth=2, markersize=8, label='GT')
if results_estimated:
    mean_errors_est = [results_estimated[idx]['mean_error'] for idx in frame_indices if idx in results_estimated]
    frame_indices_est = [idx for idx in frame_indices if idx in results_estimated]
    if frame_indices_est:
        axes[0, 0].plot(frame_indices_est, mean_errors_est, 's--', color='tab:red', linewidth=2, markersize=8, label='Estimated')
axes[0, 0].set_xlabel('Frame Index')
axes[0, 0].set_ylabel('Mean Depth Error (m)')
axes[0, 0].set_title('Mean Reprojection Error vs Frame Index')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Median error over time
axes[0, 1].plot(frame_indices, median_errors, 'o-', color='tab:orange', linewidth=2, markersize=8, label='GT')
if results_estimated:
    median_errors_est = [results_estimated[idx]['median_error'] for idx in frame_indices if idx in results_estimated]
    frame_indices_est = [idx for idx in frame_indices if idx in results_estimated]
    if frame_indices_est:
        axes[0, 1].plot(frame_indices_est, median_errors_est, 's--', color='tab:red', linewidth=2, markersize=8, label='Estimated')
axes[0, 1].set_xlabel('Frame Index')
axes[0, 1].set_ylabel('Median Depth Error (m)')
axes[0, 1].set_title('Median Reprojection Error vs Frame Index')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Valid ratio over time
axes[1, 0].plot(frame_indices, valid_ratios, 'o-', color='tab:green', linewidth=2, markersize=8, label='GT')
if results_estimated:
    valid_ratios_est = [results_estimated[idx]['n_valid'] / results_estimated[idx]['n_total'] for idx in frame_indices if idx in results_estimated]
    frame_indices_est = [idx for idx in frame_indices if idx in results_estimated]
    if frame_indices_est:
        axes[1, 0].plot(frame_indices_est, valid_ratios_est, 's--', color='tab:red', linewidth=2, markersize=8, label='Estimated')
axes[1, 0].set_xlabel('Frame Index')
axes[1, 0].set_ylabel('Valid Point Ratio')
axes[1, 0].set_title('Valid Point Ratio vs Frame Index')
axes[1, 0].set_ylim([0, 1])
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Number of valid points over time
axes[1, 1].plot(frame_indices, n_valid_list, 'o-', color='tab:purple', linewidth=2, markersize=8, label='Valid (GT)')
axes[1, 1].plot(frame_indices, n_total_list, 's--', color='tab:red', linewidth=2, markersize=8, label='Total')
if results_estimated:
    n_valid_list_est = [results_estimated[idx]['n_valid'] for idx in frame_indices if idx in results_estimated]
    frame_indices_est = [idx for idx in frame_indices if idx in results_estimated]
    if frame_indices_est:
        axes[1, 1].plot(frame_indices_est, n_valid_list_est, '^--', color='tab:cyan', linewidth=2, markersize=8, label='Valid (Estimated)')
axes[1, 1].set_xlabel('Frame Index')
axes[1, 1].set_ylabel('Number of Points')
axes[1, 1].set_title('Point Counts vs Frame Index')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

# Print table for GT poses
print("\nFrame-by-Frame Statistics (GT Poses):")
print(f"{'Frame':<8} {'Total':<10} {'Valid':<10} {'Valid%':<10} {'Mean Err':<12} {'Median Err':<12}")
print("-" * 80)
for idx in frame_indices:
    r = results[idx]
    print(f"{idx:<8} {r['n_total']:<10} {r['n_valid']:<10} {100*r['n_valid']/r['n_total']:<10.1f} "
          f"{r['mean_error']:<12.4f} {r['median_error']:<12.4f}")

# Print table for estimated poses
if results_estimated:
    print("\nFrame-by-Frame Statistics (Estimated Poses):")
    print(f"{'Frame':<8} {'Total':<10} {'Valid':<10} {'Valid%':<10} {'Mean Err':<12} {'Median Err':<12}")
    print("-" * 80)
    for idx in sorted(results_estimated.keys()):
        r = results_estimated[idx]
        print(f"{idx:<8} {r['n_total']:<10} {r['n_valid']:<10} {100*r['n_valid']/r['n_total']:<10.1f} "
              f"{r['mean_error']:<12.4f} {r['median_error']:<12.4f}")
    
    # Comparison table
    print("\nComparison: GT vs Estimated:")
    print(f"{'Frame':<8} {'GT Mean':<12} {'Est Mean':<12} {'GT Valid%':<12} {'Est Valid%':<12}")
    print("-" * 80)
    for idx in frame_indices:
        if idx in results_estimated:
            r_gt = results[idx]
            r_est = results_estimated[idx]
            print(f"{idx:<8} {r_gt['mean_error']:<12.4f} {r_est['mean_error']:<12.4f} "
                  f"{100*r_gt['n_valid']/r_gt['n_total']:<12.1f} {100*r_est['n_valid']/r_est['n_total']:<12.1f}")

In [ ]:
# Breakdown of invalid reasons across all frames
fig, axes = plt.subplots(1, 2 if results_estimated else 1, figsize=(18 if results_estimated else 12, 6))
if not results_estimated:
    axes = [axes]

# GT poses breakdown
x = np.arange(len(frame_indices))
width = 0.2

out_of_bounds_counts = [results[idx]['invalid_counts']['out_of_bounds'] for idx in frame_indices]
not_in_mask_counts = [results[idx]['invalid_counts']['not_in_mask'] for idx in frame_indices]
no_depth_counts = [results[idx]['invalid_counts']['no_depth'] for idx in frame_indices]
depth_out_of_range_counts = [results[idx]['invalid_counts']['depth_out_of_range'] for idx in frame_indices]

axes[0].bar(x - 1.5*width, out_of_bounds_counts, width, label='Out of bounds', color='red', alpha=0.7)
axes[0].bar(x - 0.5*width, not_in_mask_counts, width, label='Not in mask', color='yellow', alpha=0.7)
axes[0].bar(x + 0.5*width, no_depth_counts, width, label='No depth', color='magenta', alpha=0.7)
axes[0].bar(x + 1.5*width, depth_out_of_range_counts, width, label='Depth out of range', color='cyan', alpha=0.7)

axes[0].set_xlabel('Frame Index')
axes[0].set_ylabel('Number of Invalid Points')
axes[0].set_title('Invalid Point Breakdown by Reason (GT Poses)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(frame_indices)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Estimated poses breakdown
if results_estimated:
    frame_indices_est = sorted([idx for idx in frame_indices if idx in results_estimated])
    x_est = np.arange(len(frame_indices_est))
    
    out_of_bounds_counts_est = [results_estimated[idx]['invalid_counts']['out_of_bounds'] for idx in frame_indices_est]
    not_in_mask_counts_est = [results_estimated[idx]['invalid_counts']['not_in_mask'] for idx in frame_indices_est]
    no_depth_counts_est = [results_estimated[idx]['invalid_counts']['no_depth'] for idx in frame_indices_est]
    depth_out_of_range_counts_est = [results_estimated[idx]['invalid_counts']['depth_out_of_range'] for idx in frame_indices_est]
    
    axes[1].bar(x_est - 1.5*width, out_of_bounds_counts_est, width, label='Out of bounds', color='red', alpha=0.7)
    axes[1].bar(x_est - 0.5*width, not_in_mask_counts_est, width, label='Not in mask', color='yellow', alpha=0.7)
    axes[1].bar(x_est + 0.5*width, no_depth_counts_est, width, label='No depth', color='magenta', alpha=0.7)
    axes[1].bar(x_est + 1.5*width, depth_out_of_range_counts_est, width, label='Depth out of range', color='cyan', alpha=0.7)
    
    axes[1].set_xlabel('Frame Index')
    axes[1].set_ylabel('Number of Invalid Points')
    axes[1].set_title('Invalid Point Breakdown by Reason (Estimated Poses)')
    axes[1].set_xticks(x_est)
    axes[1].set_xticklabels(frame_indices_est)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')

fig.tight_layout()
plt.show()

# Cluster-wise Reprojection Analysis

Analyze reprojection for different clusters within a single frame. Each cluster represents a candidate transformation, and we can compare how well each cluster's transform reprojects points to the first frame.

In [ ]:
# Configuration for cluster analysis
analyze_frame_idx = 440  # Frame to analyze clusters from
# Set to None to skip cluster analysis

In [ ]:
def unpack_ragged(name: str, store: dict, dim: int = -1) -> list:
    """Unpack ragged array data from NPZ storage format."""
    data_key = f"{name}_data"
    offsets_key = f"{name}_offsets"
    lengths_key = f"{name}_lengths"
    
    if data_key not in store or offsets_key not in store or lengths_key not in store:
        return []
    
    data = store[data_key]
    offsets = store[offsets_key]
    lengths = store[lengths_key]
    
    out = []
    for off, L in zip(offsets, lengths):
        flat_data = data[off : off + L]
        
        if L == 0 or len(flat_data) == 0:
            if dim == 3:
                out.append(np.array([]).reshape(0, 3))
            elif dim == 2:
                out.append(np.array([]).reshape(0, 2))
            else:
                out.append(np.array([]))
            continue
        
        if dim == 3:
            if len(flat_data) % 3 != 0:
                out.append(flat_data)
            else:
                out.append(flat_data.reshape(-1, 3))
        elif dim == 2:
            if len(flat_data) % 2 != 0:
                out.append(flat_data)
            else:
                out.append(flat_data.reshape(-1, 2))
        else:
            out.append(flat_data)
    
    return out

def _as_py(obj):
    """Convert common numpy wrapper types to plain Python objects."""
    if obj is None:
        return None
    if isinstance(obj, np.ndarray):
        if obj.shape == ():
            return _as_py(obj.item())
        return obj.tolist()
    return obj

def clusters_for_frame(store: dict, frame_idx: int) -> list:
    """Extract clusters for a given frame from meta_data."""
    if "reg_clusters" not in store:
        return []
    arr = store["reg_clusters"]
    if not isinstance(arr, np.ndarray):
        v = arr[frame_idx]
        v = _as_py(v)
        return v if isinstance(v, list) else []
    
    v = arr[frame_idx]
    v = _as_py(v)
    if v is None:
        return []
    if isinstance(v, list):
        return [c for c in v if isinstance(c, dict)]
    if isinstance(v, dict):
        return [v]
    return []

def get_best_cluster_idx(store: dict, frame_idx: int) -> int:
    """Get the best cluster index for a frame."""
    if "reg_best_cluster_idx" in store:
        try:
            return int(np.asarray(store["reg_best_cluster_idx"])[frame_idx])
        except Exception:
            pass
    return -1

In [ ]:
# Load clusters for the specified frame
best_cluster_idx = -1  # Initialize
if analyze_frame_idx is not None and meta_data_file is not None:
    print(f"\n{'='*80}")
    print(f"CLUSTER-WISE REPROJECTION ANALYSIS FOR FRAME {analyze_frame_idx}")
    print(f"{'='*80}")
    
    # Load meta_data
    store = dict(np.load(meta_data_file, allow_pickle=True))
    
    # Get clusters for this frame
    clusters = clusters_for_frame(store, analyze_frame_idx)
    best_cluster_idx = get_best_cluster_idx(store, analyze_frame_idx)
    
    print(f"Found {len(clusters)} clusters for frame {analyze_frame_idx}")
    if best_cluster_idx >= 0:
        print(f"Best cluster index: {best_cluster_idx}")
    
    if len(clusters) == 0:
        print("No clusters found. Skipping cluster analysis.")
        cluster_results = {}
    else:
        # Load the frame to extract point cloud
        analyze_frame = load_frame_from_reader(reader, analyze_frame_idx)
        if analyze_frame is None:
            print(f"Failed to load frame {analyze_frame_idx}")
            cluster_results = {}
        else:
            # Extract full point cloud from this frame
            src_pcd_full = extract_cropped_point_cloud(analyze_frame, obj_id)
            if src_pcd_full.size == 0:
                print(f"No points extracted from frame {analyze_frame_idx}")
                cluster_results = {}
            else:
                print(f"Extracted {len(src_pcd_full)} points from frame {analyze_frame_idx}")
                
                # Get first frame pose (GT or estimated)
                if estimated_poses is not None and first_frame_idx in estimated_poses:
                    first_pose = estimated_poses[first_frame_idx]
                    pose_source = "estimated"
                else:
                    first_pose = reader.get_gt_pose(first_frame_idx)
                    pose_source = "GT"
                
                if first_pose is None:
                    print(f"Warning: No pose available for first frame")
                    cluster_results = {}
                else:
                    cluster_results = {}
                    
                    # For each cluster, compute reprojection
                    for cluster_idx, cluster in enumerate(clusters):
                        print(f"\nProcessing cluster {cluster_idx}...")
                        
                        # Get cluster transform
                        cluster_T = np.asarray(cluster.get("T", np.eye(4)), dtype=float)
                        
                        # Compute transform from current frame to first frame using this cluster
                        # T_first_curr = T_first_obj @ inv(T_curr_obj)
                        # But cluster_T is T_curr_obj (transform from object to current camera)
                        # So we need: T_first_curr = T_first_obj @ inv(cluster_T)
                        T_curr2first = first_pose @ inverse_SE3(cluster_T)
                        
                        # Compute reprojection errors
                        pts_2d, depth_errors, valid_mask, pts_dst_3d, invalid_reasons = compute_per_point_reprojection_errors(
                            src_pcd_full,
                            T_curr2first,
                            first_frame,
                            obj_id=obj_id,
                            min_depth=min_depth,
                            max_depth=max_depth,
                        )
                        
                        # Compute statistics
                        valid_errors = depth_errors[valid_mask]
                        if np.isfinite(valid_errors).any() and len(valid_errors) > 0:
                            mean_error = float(np.nanmean(valid_errors))
                            median_error = float(np.nanmedian(valid_errors))
                            std_error = float(np.nanstd(valid_errors))
                            n_valid = int(np.sum(valid_mask))
                        else:
                            mean_error = np.nan
                            median_error = np.nan
                            std_error = np.nan
                            n_valid = 0
                        
                        invalid_counts = {
                            'out_of_bounds': int(np.sum(invalid_reasons == 1)),
                            'not_in_mask': int(np.sum(invalid_reasons == 2)),
                            'no_depth': int(np.sum(invalid_reasons == 3)),
                            'depth_out_of_range': int(np.sum(invalid_reasons == 4)),
                        }
                        
                        cluster_results[cluster_idx] = {
                            'pts_2d': pts_2d,
                            'depth_errors': depth_errors,
                            'valid_mask': valid_mask,
                            'invalid_reasons': invalid_reasons,
                            'T_curr2first': T_curr2first,
                            'cluster_T': cluster_T,
                            'mean_error': mean_error,
                            'median_error': median_error,
                            'std_error': std_error,
                            'n_valid': n_valid,
                            'n_total': len(src_pcd_full),
                            'invalid_counts': invalid_counts,
                            'cluster_info': {
                                'ninliers': cluster.get('ninliers', 0),
                                'mean_res': cluster.get('mean_res', np.nan),
                                'score': cluster.get('score', np.nan),
                                'reproj_error': cluster.get('reproj_error', np.nan),
                            }
                        }
                        
                        print(f"  Valid points: {n_valid}/{len(src_pcd_full)} ({100*n_valid/len(src_pcd_full):.1f}%)")
                        print(f"  Mean depth error: {mean_error:.4f}m")
                        print(f"  Cluster ninliers: {cluster.get('ninliers', 0)}, mean_res: {cluster.get('mean_res', np.nan):.6f}")
else:
    print("Skipping cluster analysis (analyze_frame_idx is None or meta_data not available)")
    cluster_results = {}

In [ ]:
# Visualize reprojection for each cluster
if cluster_results:
    print(f"\n{'='*80}")
    print(f"VISUALIZING CLUSTER REPROJECTIONS")
    print(f"{'='*80}")
    
    for cluster_idx, result in cluster_results.items():
        cluster_info = result['cluster_info']
        is_best = (cluster_idx == best_cluster_idx)
        
        fig = plot_reprojection_visualization(
            first_frame,
            result['pts_2d'],
            result['depth_errors'],
            result['valid_mask'],
            result['invalid_reasons'],
            analyze_frame_idx,
            {
                'mean_error': result['mean_error'],
                'median_error': result['median_error'],
                'n_valid': result['n_valid'],
                'n_total': result['n_total'],
            },
            title_suffix=f" - Cluster {cluster_idx}{' (BEST)' if is_best else ''} (ninliers={cluster_info['ninliers']}, mean_res={cluster_info['mean_res']:.6f})",
        )
        plt.show()

In [ ]:
# Summary comparison of all clusters
if cluster_results:
    print(f"\n{'='*80}")
    print(f"CLUSTER COMPARISON SUMMARY")
    print(f"{'='*80}")
    
    cluster_indices = sorted(cluster_results.keys())
    
    # Extract statistics
    mean_errors = [cluster_results[idx]['mean_error'] for idx in cluster_indices]
    median_errors = [cluster_results[idx]['median_error'] for idx in cluster_indices]
    valid_ratios = [cluster_results[idx]['n_valid'] / cluster_results[idx]['n_total'] for idx in cluster_indices]
    n_valid_list = [cluster_results[idx]['n_valid'] for idx in cluster_indices]
    ninliers_list = [cluster_results[idx]['cluster_info']['ninliers'] for idx in cluster_indices]
    mean_res_list = [cluster_results[idx]['cluster_info']['mean_res'] for idx in cluster_indices]
    reproj_error_list = [cluster_results[idx]['cluster_info']['reproj_error'] for idx in cluster_indices]
    
    # Create comparison plots
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    x = np.arange(len(cluster_indices))
    
    # Mean reprojection error
    axes[0, 0].bar(x, mean_errors, color='tab:blue', alpha=0.7)
    if best_cluster_idx >= 0 and best_cluster_idx in cluster_indices:
        best_x = cluster_indices.index(best_cluster_idx)
        axes[0, 0].axvline(best_x, color='red', linestyle='--', linewidth=2, label='Best cluster')
    axes[0, 0].set_xlabel('Cluster Index')
    axes[0, 0].set_ylabel('Mean Depth Error (m)')
    axes[0, 0].set_title('Mean Reprojection Error per Cluster')
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(cluster_indices)
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3, axis='y')
    
    # Median reprojection error
    axes[0, 1].bar(x, median_errors, color='tab:orange', alpha=0.7)
    if best_cluster_idx >= 0 and best_cluster_idx in cluster_indices:
        best_x = cluster_indices.index(best_cluster_idx)
        axes[0, 1].axvline(best_x, color='red', linestyle='--', linewidth=2, label='Best cluster')
    axes[0, 1].set_xlabel('Cluster Index')
    axes[0, 1].set_ylabel('Median Depth Error (m)')
    axes[0, 1].set_title('Median Reprojection Error per Cluster')
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels(cluster_indices)
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3, axis='y')
    
    # Valid ratio
    axes[0, 2].bar(x, valid_ratios, color='tab:green', alpha=0.7)
    if best_cluster_idx >= 0 and best_cluster_idx in cluster_indices:
        best_x = cluster_indices.index(best_cluster_idx)
        axes[0, 2].axvline(best_x, color='red', linestyle='--', linewidth=2, label='Best cluster')
    axes[0, 2].set_xlabel('Cluster Index')
    axes[0, 2].set_ylabel('Valid Point Ratio')
    axes[0, 2].set_title('Valid Point Ratio per Cluster')
    axes[0, 2].set_ylim([0, 1])
    axes[0, 2].set_xticks(x)
    axes[0, 2].set_xticklabels(cluster_indices)
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3, axis='y')
    
    # Number of inliers
    axes[1, 0].bar(x, ninliers_list, color='tab:purple', alpha=0.7)
    if best_cluster_idx >= 0 and best_cluster_idx in cluster_indices:
        best_x = cluster_indices.index(best_cluster_idx)
        axes[1, 0].axvline(best_x, color='red', linestyle='--', linewidth=2, label='Best cluster')
    axes[1, 0].set_xlabel('Cluster Index')
    axes[1, 0].set_ylabel('Number of Inliers')
    axes[1, 0].set_title('Cluster Inliers Count')
    axes[1, 0].set_xticks(x)
    axes[1, 0].set_xticklabels(cluster_indices)
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    
    # Mean residual
    axes[1, 1].bar(x, mean_res_list, color='tab:red', alpha=0.7)
    if best_cluster_idx >= 0 and best_cluster_idx in cluster_indices:
        best_x = cluster_indices.index(best_cluster_idx)
        axes[1, 1].axvline(best_x, color='red', linestyle='--', linewidth=2, label='Best cluster')
    axes[1, 1].set_xlabel('Cluster Index')
    axes[1, 1].set_ylabel('Mean Residual')
    axes[1, 1].set_title('Cluster Mean Residual')
    axes[1, 1].set_xticks(x)
    axes[1, 1].set_xticklabels(cluster_indices)
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    
    # Reprojection error (if available)
    if any(np.isfinite(reproj_error_list)):
        axes[1, 2].bar(x, reproj_error_list, color='tab:cyan', alpha=0.7)
        if best_cluster_idx >= 0 and best_cluster_idx in cluster_indices:
            best_x = cluster_indices.index(best_cluster_idx)
            axes[1, 2].axvline(best_x, color='red', linestyle='--', linewidth=2, label='Best cluster')
        axes[1, 2].set_xlabel('Cluster Index')
        axes[1, 2].set_ylabel('Reprojection Error')
        axes[1, 2].set_title('Cluster Reprojection Error (from register)')
        axes[1, 2].set_xticks(x)
        axes[1, 2].set_xticklabels(cluster_indices)
        axes[1, 2].legend()
        axes[1, 2].grid(True, alpha=0.3, axis='y')
    else:
        axes[1, 2].text(0.5, 0.5, 'No reprojection errors\navailable in clusters', 
                        ha='center', va='center', transform=axes[1, 2].transAxes)
        axes[1, 2].set_title('Cluster Reprojection Error')
    
    fig.suptitle(f'Cluster Comparison - Frame {analyze_frame_idx}', fontsize=14)
    fig.tight_layout()
    plt.show()
    
    # Print comparison table
    print("\nCluster-by-Cluster Statistics:")
    print(f"{'Clust':<8} {'Ninl':<8} {'MeanRes':<12} {'ReprojErr':<12} {'Valid':<8} {'Valid%':<10} {'MeanErr':<12} {'MedErr':<12}")
    print("-" * 100)
    for idx in cluster_indices:
        r = cluster_results[idx]
        ci = r['cluster_info']
        marker = " <-- BEST" if idx == best_cluster_idx else ""
        print(f"{idx:<8} {ci['ninliers']:<8} {ci['mean_res']:<12.6f} {ci['reproj_error']:<12.4f} "
              f"{r['n_valid']:<8} {100*r['n_valid']/r['n_total']:<10.1f} "
              f"{r['mean_error']:<12.4f} {r['median_error']:<12.4f}{marker}")

In [ ]:
# Plot cluster summary with errors and residuals (similar to plot_clustered_registration_stats.py)
if cluster_results:
    print(f"\n{'='*80}")
    print(f"CLUSTER ERRORS AND RESIDUALS SUMMARY")
    print(f"{'='*80}")
    
    cluster_indices = sorted(cluster_results.keys())
    
    # Extract cluster metrics
    ninliers_list = [cluster_results[idx]['cluster_info']['ninliers'] for idx in cluster_indices]
    mean_res_list = [cluster_results[idx]['cluster_info']['mean_res'] for idx in cluster_indices]
    score_list = [cluster_results[idx]['cluster_info']['score'] for idx in cluster_indices]
    reproj_error_list = [cluster_results[idx]['cluster_info']['reproj_error'] for idx in cluster_indices]
    
    # Extract reprojection metrics
    mean_errors = [cluster_results[idx]['mean_error'] for idx in cluster_indices]
    median_errors = [cluster_results[idx]['median_error'] for idx in cluster_indices]
    std_errors = [cluster_results[idx]['std_error'] for idx in cluster_indices]
    n_valid_list = [cluster_results[idx]['n_valid'] for idx in cluster_indices]
    
    # Determine number of plots needed
    has_reproj_error = any(np.isfinite(reproj_error_list))
    has_score = any(np.isfinite(score_list))
    
    n_plots = 5  # ninliers, mean_res, mean_reproj_error, median_reproj_error, n_valid
    if has_reproj_error:
        n_plots += 1
    if has_score:
        n_plots += 1
    
    fig, axs = plt.subplots(n_plots, 1, figsize=(12, 2.5 * n_plots), sharex=True)
    if n_plots == 1:
        axs = [axs]
    
    x = np.arange(len(cluster_indices))
    
    row = 0
    
    # 1. Number of inliers
    axs[row].bar(x, ninliers_list, color="tab:blue", alpha=0.8)
    if best_cluster_idx >= 0 and best_cluster_idx in cluster_indices:
        best_x = cluster_indices.index(best_cluster_idx)
        axs[row].axvline(best_x, color="red", linestyle="--", alpha=0.7, linewidth=1.5, label="Best cluster")
    axs[row].set_ylabel("ninliers")
    axs[row].set_title(f"Cluster Summary - Frame {analyze_frame_idx}")
    axs[row].legend()
    axs[row].grid(True, alpha=0.3, axis='y')
    row += 1
    
    # 2. Mean residual
    axs[row].plot(x, mean_res_list, "o-", color="tab:orange", alpha=0.9, linewidth=2, markersize=8)
    if best_cluster_idx >= 0 and best_cluster_idx in cluster_indices:
        best_x = cluster_indices.index(best_cluster_idx)
        axs[row].axvline(best_x, color="red", linestyle="--", alpha=0.7, linewidth=1.5, label="Best cluster")
    axs[row].set_ylabel("mean_res (cluster)")
    axs[row].legend()
    axs[row].grid(True, alpha=0.3)
    row += 1
    
    # 3. Score (if available)
    if has_score:
        axs[row].plot(x, score_list, "o-", color="tab:green", alpha=0.9, linewidth=2, markersize=8)
        if best_cluster_idx >= 0 and best_cluster_idx in cluster_indices:
            best_x = cluster_indices.index(best_cluster_idx)
            axs[row].axvline(best_x, color="red", linestyle="--", alpha=0.7, linewidth=1.5, label="Best cluster")
        axs[row].set_ylabel("score")
        axs[row].legend()
        axs[row].grid(True, alpha=0.3)
        row += 1
    
    # 4. Reprojection error from registration (if available)
    if has_reproj_error:
        axs[row].plot(x, reproj_error_list, "o-", color="tab:pink", alpha=0.9, linewidth=2, markersize=8)
        if best_cluster_idx >= 0 and best_cluster_idx in cluster_indices:
            best_x = cluster_indices.index(best_cluster_idx)
            axs[row].axvline(best_x, color="red", linestyle="--", alpha=0.7, linewidth=1.5, label="Best cluster")
        axs[row].set_ylabel("reproj_error (from register)")
        axs[row].legend()
        axs[row].grid(True, alpha=0.3)
        row += 1
    
    # 5. Mean reprojection error (to first frame)
    axs[row].plot(x, mean_errors, "o-", color="tab:purple", alpha=0.9, linewidth=2, markersize=8)
    if best_cluster_idx >= 0 and best_cluster_idx in cluster_indices:
        best_x = cluster_indices.index(best_cluster_idx)
        axs[row].axvline(best_x, color="red", linestyle="--", alpha=0.7, linewidth=1.5, label="Best cluster")
    axs[row].set_ylabel("mean reproj error (m)")
    axs[row].legend()
    axs[row].grid(True, alpha=0.3)
    row += 1
    
    # 6. Median reprojection error (to first frame)
    axs[row].plot(x, median_errors, "o-", color="tab:cyan", alpha=0.9, linewidth=2, markersize=8)
    if best_cluster_idx >= 0 and best_cluster_idx in cluster_indices:
        best_x = cluster_indices.index(best_cluster_idx)
        axs[row].axvline(best_x, color="red", linestyle="--", alpha=0.7, linewidth=1.5, label="Best cluster")
    axs[row].set_ylabel("median reproj error (m)")
    axs[row].legend()
    axs[row].grid(True, alpha=0.3)
    row += 1
    
    # 7. Number of valid points
    axs[row].bar(x, n_valid_list, color="tab:red", alpha=0.8)
    if best_cluster_idx >= 0 and best_cluster_idx in cluster_indices:
        best_x = cluster_indices.index(best_cluster_idx)
        axs[row].axvline(best_x, color="red", linestyle="--", alpha=0.7, linewidth=1.5, label="Best cluster")
    axs[row].set_ylabel("n_valid points")
    axs[row].set_xlabel("cluster candidate index")
    axs[row].set_xticks(x)
    axs[row].set_xticklabels(cluster_indices)
    axs[row].legend()
    axs[row].grid(True, alpha=0.3, axis='y')
    
    # Add best cluster indicators
    best_metrics = []
    if np.isfinite(mean_errors).any():
        best_mean_err_idx = cluster_indices[np.nanargmin(mean_errors)]
        if best_mean_err_idx != best_cluster_idx:
            best_metrics.append(f"min mean_err={cluster_indices.index(best_mean_err_idx)}")
    if np.isfinite(median_errors).any():
        best_med_err_idx = cluster_indices[np.nanargmin(median_errors)]
        if best_med_err_idx != best_cluster_idx:
            best_metrics.append(f"min med_err={cluster_indices.index(best_med_err_idx)}")
    if np.isfinite(mean_res_list).any():
        best_res_idx = cluster_indices[np.nanargmin(mean_res_list)]
        if best_res_idx != best_cluster_idx:
            best_metrics.append(f"min mean_res={cluster_indices.index(best_res_idx)}")
    
    if best_metrics:
        extra_str = f" | {'; '.join(best_metrics)}"
        axs[0].set_title(f"Cluster Summary - Frame {analyze_frame_idx} (best_idx={best_cluster_idx}){extra_str}")
    else:
        axs[0].set_title(f"Cluster Summary - Frame {analyze_frame_idx} (best_idx={best_cluster_idx})")
    
    fig.tight_layout()
    plt.show()

In [ ]:
# Additional: Plot comparison with GT pose if available
if cluster_results:
    # Try to get GT poses for comparison
    gt_pose_curr = reader.get_gt_pose(analyze_frame_idx)
    gt_pose_first = reader.get_gt_pose(first_frame_idx)
    
    if gt_pose_curr is not None and gt_pose_first is not None:
        print(f"\n{'='*80}")
        print(f"CLUSTER COMPARISON WITH GT POSE")
        print(f"{'='*80}")
        
        # Compute GT transform from current to first frame
        T_gt_curr2first = gt_pose_first @ inverse_SE3(gt_pose_curr)
        
        # Helper function to compute pose error
        def pose_error(T_est, T_gt):
            """Return (translation_error, rotation_error_deg) for T_est vs T_gt."""
            T_err = T_est @ inverse_SE3(T_gt)
            t_err = T_err[:3, 3]
            R_err = T_err[:3, :3]
            # Compute rotation angle
            cos = (np.trace(R_err) - 1.0) * 0.5
            cos = float(np.clip(cos, -1.0, 1.0))
            rot_err_deg = float(np.degrees(np.arccos(cos)))
            return float(np.linalg.norm(t_err)), rot_err_deg
        
        cluster_indices = sorted(cluster_results.keys())
        trans_errors = []
        rot_errors = []
        
        for idx in cluster_indices:
            T_cluster = cluster_results[idx]['T_curr2first']
            te, re = pose_error(T_cluster, T_gt_curr2first)
            trans_errors.append(te)
            rot_errors.append(re)
        
        trans_errors = np.array(trans_errors)
        rot_errors = np.array(rot_errors)
        
        # Find best cluster by GT error
        best_gt_idx = -1
        if np.isfinite(trans_errors).any():
            best_gt_idx = cluster_indices[np.nanargmin(trans_errors)]
        
        # Create comparison plot
        fig, axs = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
        x = np.arange(len(cluster_indices))
        
        # Translation error
        axs[0].plot(x, trans_errors, "o-", color="tab:purple", alpha=0.9, linewidth=2, markersize=8, label="Translation error")
        if best_cluster_idx >= 0 and best_cluster_idx in cluster_indices:
            best_x = cluster_indices.index(best_cluster_idx)
            axs[0].axvline(best_x, color="red", linestyle="--", alpha=0.7, linewidth=1.5, label="Best cluster (selected)")
        if best_gt_idx >= 0 and best_gt_idx in cluster_indices:
            best_gt_x = cluster_indices.index(best_gt_idx)
            axs[0].axvline(best_gt_x, color="magenta", linestyle=":", alpha=0.8, linewidth=1.4, label="Best cluster (GT)")
        axs[0].set_ylabel("Translation error (m)")
        axs[0].set_title(f"Cluster Pose Error vs GT - Frame {analyze_frame_idx}\n"
                        f"black=selected best_idx ({best_cluster_idx}), magenta=min trans error ({best_gt_idx})")
        axs[0].legend()
        axs[0].grid(True, alpha=0.3)
        
        # Rotation error
        axs[1].plot(x, rot_errors, "o-", color="tab:red", alpha=0.9, linewidth=2, markersize=8, label="Rotation error")
        if best_cluster_idx >= 0 and best_cluster_idx in cluster_indices:
            best_x = cluster_indices.index(best_cluster_idx)
            axs[1].axvline(best_x, color="red", linestyle="--", alpha=0.7, linewidth=1.5, label="Best cluster (selected)")
        if best_gt_idx >= 0 and best_gt_idx in cluster_indices:
            best_gt_x = cluster_indices.index(best_gt_idx)
            axs[1].axvline(best_gt_x, color="magenta", linestyle=":", alpha=0.8, linewidth=1.4, label="Best cluster (GT)")
        axs[1].set_ylabel("Rotation error (deg)")
        axs[1].set_xlabel("cluster candidate index")
        axs[1].set_xticks(x)
        axs[1].set_xticklabels(cluster_indices)
        axs[1].legend()
        axs[1].grid(True, alpha=0.3)
        
        fig.tight_layout()
        plt.show()
        
        # Print comparison table
        print("\nCluster Pose Error vs GT:")
        print(f"{'Clust':<8} {'Trans Err':<12} {'Rot Err':<12} {'Mean Reproj':<12} {'Ninl':<8}")
        print("-" * 80)
        for idx in cluster_indices:
            r = cluster_results[idx]
            ci = r['cluster_info']
            trans_marker = " <-- GT BEST" if idx == best_gt_idx else ""
            select_marker = " <-- SELECTED" if idx == best_cluster_idx else ""
            marker = trans_marker + select_marker
            print(f"{idx:<8} {trans_errors[cluster_indices.index(idx)]:<12.4f} "
                  f"{rot_errors[cluster_indices.index(idx)]:<12.2f} "
                  f"{r['mean_error']:<12.4f} {ci['ninliers']:<8}{marker}")
    else:
        print("GT poses not available for comparison")